In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_vco_map_S5_20261020_170000.csv"; None = newest map

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.vco_lut import VcoLUT

if RESULT_CSV is None:
    RESULT_CSV = sorted(PLL_DIR.glob("pll_vco_map_*.csv"), key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
CODES = ["vco_tune_coarse", "vco_current_min", "vco_current_max"]
MIN_DV = 0.003  # [V] Kvco only between locked points at least this far apart in Vctrl (pll_test_current_max.MAP_MIN_DV)
pts = pd.read_csv(RESULT_CSV).sort_values(CODES + ["f_vco"]).reset_index(drop=True)
pts["setting"] = [f"c{r.vco_tune_coarse} min {r.vco_current_min} max {r.vco_current_max}" for r in pts.itertuples()]
locked = pts[(pts["locked"] == 1) & pts["vctrl_meas"].notna()]
lut = VcoLUT(csv=PLL_DIR.parents[0] / "vco" / pts["lut_csv"].iloc[0])
print(f"{RESULT_CSV.name}: {pts['setting'].nunique()} settings, {len(pts)} points, {len(locked)} locked; "
      f"loop filter Icp {pts['set_current'].iloc[0]}, C1 {pts['set_c1'].iloc[0]}, C2 {pts['set_c2'].iloc[0]}, "
      f"R1 {pts['set_r1'].iloc[0]}; table {lut.path.name}")


def kvco_points(d):
    """Measured |Kvco| of one setting's locked points (sorted by Vctrl): midpoints of neighbours >= MIN_DV apart."""
    d = d.sort_values("vctrl_meas")
    out, i = [], 0
    v, f = d["vctrl_meas"].to_numpy(), d["f_vco"].to_numpy()
    while i < len(v) - 1:
        j = i + 1
        while j < len(v) and v[j] - v[i] < MIN_DV:
            j += 1
        if j == len(v):
            break
        out.append(((v[i] + v[j]) / 2, (f[i] + f[j]) / 2, abs(f[j] - f[i]) / (v[j] - v[i])))
        i = j
    return pd.DataFrame(out, columns=["vctrl", "f_vco", "kvco"])


# Per setting: where it locked, the measured Vctrl range, the measured |Kvco| range and the table's Vctrl error
summary = []
for setting, d in pts.groupby("setting", sort=False):
    lk = locked[locked["setting"] == setting]
    k = kvco_points(lk)
    summary.append({
        "setting": setting, "points": len(d), "locked": len(lk),
        "f locked [MHz]": f"{lk['f_vco'].min() / 1e6:.0f}-{lk['f_vco'].max() / 1e6:.0f}" if len(lk) else "-",
        "Vctrl [V]": f"{lk['vctrl_meas'].min():.3f}-{lk['vctrl_meas'].max():.3f}" if len(lk) else "-",
        "max |Kvco| [GHz/V]": k["kvco"].max() / 1e9 if len(k) else np.nan,
        "Vctrl - table [mV] (median)": (lk["vctrl_meas"] - lk["vctrl_table"]).median() * 1e3,
    })
pd.DataFrame(summary).round(2)

pll_vco_map_S5_20261020_180756.csv: 36 settings, 720 points, 552 locked; loop filter Icp 1, C1 7, C2 7, R1 3; table vco_lut_S5_20261001_161638.csv


,setting,points,locked,f locked [MHz],Vctrl [V],max |Kvco| [GHz/V],Vctrl - table [mV] (median)
0,c0 min 9 max 0,20,18,2281-9583,0.397-0.610,47.30,-19.48
1,c0 min 9 max 8,20,17,2114-6598,0.419-0.642,170.35,-18.76
2,c0 min 9 max 15,20,17,1903-2363,0.394-0.770,4.79,-42.37
3,c1 min 9 max 0,20,16,1673-5359,0.417-0.622,28.77,-20.33
4,c1 min 9 max 8,20,18,1614-4559,0.393-0.627,43.85,-21.52
5,c1 min 9 max 15,20,15,1499-1797,0.426-0.655,1.97,-38.80
6,c3 min 9 max 0,20,17,1183-2699,0.408-0.645,12.56,-21.07
7,c3 min 9 max 8,20,15,1173-2578,0.408-0.643,10.13,-22.40
8,c3 min 9 max 15,20,15,1129-1340,0.427-0.664,1.49,-39.46
9,c5 min 9 max 0,20,17,949-1759,0.459-0.664,6.86,-20.95


In [3]:
INK, GRID = "#52514e", "#e4e3df"
GROUPS = list(dict.fromkeys(zip(pts["vco_tune_coarse"], pts["vco_current_min"])))  # one panel per coarse / min
N_COLS = 2
N_ROWS = (len(GROUPS) + N_COLS - 1) // N_COLS


def max_color(code):
    """Colour of a vco_current_max code (0: most current, blue; 15: least, yellow)."""
    return sample_colorscale("Viridis", [code / 15])[0]


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()


# f_vco against Vctrl per setting, one panel per coarse / min, colour = max code. Markers + line: measured in closed
# loop (locked points); dashed: the lookup table's curve (open loop, 2026-10-01). The horizontal gap between the two
# is the table's Vctrl error.
fig = make_subplots(rows=N_ROWS, cols=N_COLS, horizontal_spacing=0.08, vertical_spacing=0.06,
                    subplot_titles=[f"c{c} min {m}" for c, m in GROUPS])
for n, (coarse, cur_min) in enumerate(GROUPS):
    row, col = n // N_COLS + 1, n % N_COLS + 1
    for code, d in locked[(locked["vco_tune_coarse"] == coarse) & (locked["vco_current_min"] == cur_min)].groupby(
            "vco_current_max"):
        d = d.sort_values("vctrl_meas")
        fig.add_scatter(x=d["vctrl_meas"], y=d["f_vco"] / 1e6, mode="lines+markers", name=f"max {code}",
                        legendgroup=f"max {code}", showlegend=n == 0, line=dict(color=max_color(code)),
                        marker=dict(size=5), row=row, col=col,
                        hovertemplate=f"c{coarse} min {cur_min} max {code}: " + "%{x:.3f} V, %{y:.0f} MHz<extra></extra>")
        vs, fs = lut.curve({"vco_tune_coarse": coarse, "vco_current_min": cur_min, "vco_current_max": code})
        fig.add_scatter(x=vs, y=np.array(fs) / 1e6, mode="lines", showlegend=False, legendgroup=f"max {code}",
                        line=dict(color=max_color(code), dash="dash", width=1), row=row, col=col, hoverinfo="skip")
fig.update_layout(title="VCO curves measured in closed loop (markers) and the lookup table (dashed)",
                  template="plotly_white", height=320 * N_ROWS, legend=dict(orientation="h", y=-0.04))
fig.update_xaxes(title="Vctrl [V]", range=[0.35, 0.75], gridcolor=GRID)
fig.update_yaxes(title="f_vco [MHz]", gridcolor=GRID)
save(fig, "curves")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_vco_map_S5_20261020_180756_curves.html and .png


In [4]:
# Measured |Kvco| against Vctrl per setting (between neighbouring locked points >= MIN_DV apart), same panels. This is
# what steps 2 and 3 pick their points from (VcoMap.pick).
fig = make_subplots(rows=N_ROWS, cols=N_COLS, horizontal_spacing=0.08, vertical_spacing=0.06,
                    subplot_titles=[f"c{c} min {m}" for c, m in GROUPS])
for n, (coarse, cur_min) in enumerate(GROUPS):
    row, col = n // N_COLS + 1, n % N_COLS + 1
    for code, d in locked[(locked["vco_tune_coarse"] == coarse) & (locked["vco_current_min"] == cur_min)].groupby(
            "vco_current_max"):
        k = kvco_points(d)
        fig.add_scatter(x=k["vctrl"], y=k["kvco"] / 1e9, mode="lines+markers", name=f"max {code}",
                        legendgroup=f"max {code}", showlegend=n == 0, line=dict(color=max_color(code)),
                        marker=dict(size=5), row=row, col=col, customdata=k["f_vco"] / 1e6,
                        hovertemplate=f"c{coarse} min {cur_min} max {code}: "
                                      + "%{x:.3f} V, %{customdata:.0f} MHz: %{y:.2f} GHz/V<extra></extra>")
fig.update_layout(title="|Kvco| measured in closed loop", template="plotly_white", height=320 * N_ROWS,
                  legend=dict(orientation="h", y=-0.04))
fig.update_xaxes(title="Vctrl [V]", range=[0.35, 0.75], gridcolor=GRID)
fig.update_yaxes(title="|Kvco| [GHz/V]", rangemode="tozero", gridcolor=GRID)
save(fig, "kvco")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_vco_map_S5_20261020_180756_kvco.html and .png


In [5]:
# Lock map: every requested point per setting (circle: locked, x: not locked), coloured by the measured Vctrl. Shows the
# frequency range each setting can lock to with the map's (conservative) filter.
order = list(dict.fromkeys(pts["setting"]))
fig = go.Figure()
for is_locked, symbol, name in ((1, "circle", "locked"), (0, "x", "not locked")):
    d = pts[pts["locked"] == is_locked]
    fig.add_scatter(x=d["f_vco"] / 1e6, y=d["setting"], mode="markers", name=name,
                    marker=dict(symbol=symbol, size=8, color=d["vctrl_meas"], colorscale="Viridis", cmin=0.35,
                                cmax=0.75, showscale=is_locked == 1, colorbar=dict(title="Vctrl [V]")),
                    customdata=d["vctrl_meas"],
                    hovertemplate="%{y}: %{x:.0f} MHz, Vctrl %{customdata:.3f} V<extra>" + name + "</extra>")
fig.update_layout(title="Lock map of the closed-loop VCO map", template="plotly_white", height=22 * len(order) + 200,
                  legend=dict(orientation="h", y=-0.08))
fig.update_xaxes(title="f_vco [MHz]", type="log", gridcolor=GRID)
fig.update_yaxes(categoryorder="array", categoryarray=order[::-1], gridcolor=GRID)
save(fig, "lock_map")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_vco_map_S5_20261020_180756_lock_map.html and .png
